In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [22]:
import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python" or "json" or "regex",
        "solution_criteria": "Criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

In [23]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
# Functions to validate the output structure
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [24]:
def grade_by_model(test_case, output):
    prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI generated solution:
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>
    
    Solution to Evaluate:
    <solution>
    {output}
    </solution>
    
    Solution Criteria:
    <solution_criteria>
    {test_case["solution_criteria"]}
    </solution_criteria>

    Output Format
    Provide your evaluation as a structured JSON object with the following format:
    - "strengths": A list of strengths of the solution.
    - "weaknesses": An array of 1-3 areas for improvement.
    - "reasoning": A detailed explanation of your evaluation, including why you assigned the score.
    - "score": An overall score for the solution on a scale of 1 to 10.
    
    Respond with only the JSON object, without any additional text or explanation.
    Example output:
    {{
        "strengths": ["Strength 1", "Strength 2"],
        "weaknesses": ["Weakness 1", "Weakness 2"], 
        "reasoning": "Detailed explanation of the evaluation and reasoning behind the score.",
        "score": 7
    }}
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)


def run_prompt(test_case):
    # Merges the prompt and test case input, then returns the result
    prompt = f"""
    Please solve the following task:
    
    {test_case["task"]}
    
    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    """

    messages = []
    add_user_message(messages, prompt)
    response = chat(messages)
    return response


def run_test_case(test_case):
    output = run_prompt(test_case)

    # todo grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(output, test_case)
    
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

from statistics import mean

def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    return results

In [25]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

Average Score: 3.5


In [26]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n```python\nimport anthropic\nimport json\nimport re\nimport sys\n\ndef extract_ec2_instances(template_content: str) -> dict:\n    \"\"\"\n    Parse a CloudFormation template to extract EC2 instance logical IDs and their instance types.\n    \"\"\"\n    client = anthropic.Anthropic()\n    \n    message = client.messages.create(\n        model=\"claude-3-5-sonnet-20241022\",\n        max_tokens=1024,\n        messages=[\n            {\n                \"role\": \"user\",\n                \"content\": f\"\"\"Parse the following AWS CloudFormation template and extract all EC2 instance logical IDs and their instance types.\n\nReturn the result as a JSON object where keys are logical IDs and values are instance types.\n\nCloudFormation Template:\n{template_content}\n\nReturn only valid JSON, no other text.\"\"\"\n            }\n        ]\n    )\n    \n    response_text = message.content[0].text\n    result = json.loads(response_text)\n    return result\n\n\ndef extract